# FPP13 — Joint Training: Minutiae Encoder + Siamese Migration Matcher (3 arms; 3 repetitions)

## Deployment (unchanged from FPP12)
1. **Batch migration, once:** minutiae template → minutiae encoder → Siamese tower (minutiae offset) → migration template.
2. **First login:** probe image → frozen image model (FPP04L) → Siamese tower (image offset) → cosine with the migration template.
3. **On a match:** re-enrol with the normal image embedding; encoder and tower are no longer used. The image model and image templates never change.

## The question
FPP12's winner (`siamese`, first-login validation EER 10.59 %) was trained on **frozen, saved** minutiae embeddings of the clean templates: no augmentation, so it over-fitted within ~20 epochs. FPP13 feeds the tower **freshly augmented minutiae templates** through the encoder, and then also lets the **encoder learn together with the tower**.

## Arms (each adds one thing to the previous one)
| Arm | Encoder (FPP11 `best_plus_mm`) | Siamese tower (FPP12 `siamese`) | Minutiae augmentation | Loss | LR |
|---|---|---|---|---|---|
| **`tower_aug`** | frozen | trained | **on** | pair BCE + pull (FPP12 winner) | 1e-3 (as FPP12) |
| **`joint`** | **trained** | trained | on | pair BCE + pull | 3e-4 (as FPP11) |
| **`joint_mm`** | trained | trained | on | pair BCE + pull **+ 0.5 × minutiae-vs-minutiae pair BCE** | 3e-4 |

All arms start from the FPP11 `best_plus_mm` encoder and the FPP12 `siamese` tower of the same repetition (both checked against their saved outputs). Augmentation = the FPP09 minutiae augmentation incl. partial crop; the image side always uses the clean image embeddings.

## Candidates and decision
- **Candidates:** the 3 arms + **FPP12 `siamese`** (previous winner, re-scored from its saved outputs, not re-run).
- **Decision on first-login cross-modal validation only** (source-balanced EER; ties within 0.01 → AUC). Test and FingerDOS incl. F1 reported for every row, never used.
- **References:** image vs image (after re-enrolment), minutiae vs minutiae (FPP09), FingerJetFX + BOZORTH3.

## Training
Source-balanced sampler (DB1 / DB2 / DB1×DB2), 64 pairs per batch, 12 000 pairs per epoch; AdamW, weight decay 1e-4, clip 1.0; ≤ 60 epochs; early stopping (patience 10), LR × 0.3 after 3, best epoch — on the **first-login source-balanced validation EER**.

**Hardware:** A100; ≈ 15–30 min per run (the encoder runs on every augmented template). **Needs:** FPP05, FPP09 manifests, FPP11 `best_plus_mm` and FPP12 `siamese` run folders, fpp_lib ≥ 1.12.

In [ ]:
%pip install -q openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load the shared libraries from output/00_code/ (fpf_lib from FPF00, fpp_lib installed by FPP01), so every FPP kernel uses the same version.
import shutil, sys, importlib
from pathlib import Path
def _find_folder(name, root=Path('/content/drive/MyDrive'), max_depth=4):
    frontier = [root]
    for _ in range(max_depth + 1):
        nxt = []
        for d in frontier:
            try: kids = [c for c in d.iterdir() if c.is_dir()]
            except OSError: continue
            for c in kids:
                if c.name == name: return c
                nxt.append(c)
        frontier = nxt
    return None
ROOT = _find_folder('FingerPrint-Final')
if ROOT is None: raise FileNotFoundError("'FingerPrint-Final' not found under My Drive")
CODE = ROOT / 'output' / '00_code'
for m in ('fpf_lib.py', 'fpf_tf.py', 'fpp_lib.py'):
    src = CODE / m
    if not src.is_file(): raise FileNotFoundError(f'{src} missing: run ' + ('FPP01' if m == 'fpp_lib.py' else 'FPF00') + ' first')
    shutil.copy2(src, Path('/content') / m)
sys.path.insert(0, '/content')
import fpf_lib as L; importlib.reload(L)
import fpf_tf as T; importlib.reload(T)
import fpp_lib as P; importlib.reload(P)
import numpy as np, pandas as pd, json, time, os, gc
if tuple(int(x) for x in P.VERSION.split('.')) < (1, 12): raise RuntimeError(f'fpp_lib {P.VERSION} in output/00_code is too old: upload fpp_lib 1.12')
ENV = L.env_info(); print('fpp_lib', P.VERSION, '|', json.dumps(ENV, indent=1))

## 1. Configuration; start models: FPP11 `best_plus_mm` encoder, FPP12 `siamese` tower; frozen FPP04L image embeddings

In [ ]:
import h5py, math, shutil, subprocess, tensorflow as tf
import matplotlib.pyplot as plt
KERNEL = 'FPP13'; ws = L.workspace(); KDIR = P.kernel_dir(ws, KERNEL); FIG = KDIR / 'figures'
FPP11_DIR = P.kernel_dir(ws, 'FPP11'); FPP12_DIR = P.kernel_dir(ws, 'FPP12')
REPS = L.repetitions(ws); REPS_TO_RUN = REPS
ARMS = ['tower_aug', 'joint', 'joint_mm']; ARMS_TO_RUN = ARMS
ARM_CFG = {'tower_aug': dict(train_encoder=False, mm=False, lr=1e-3), 'joint': dict(train_encoder=True, mm=False, lr=3e-4), 'joint_mm': dict(train_encoder=True, mm=True, lr=3e-4)}
PREV = 'FPP12 siamese (previous winner)'
LAMBDA_PULL = 1.0; LAMBDA_MM = 0.5; ADAPTER_HIDDEN = 2048; ADAPTER_DROPOUT = 0.1
EPOCHS = 60; WEIGHT_DECAY = 1e-4; CLIPNORM = 1.0; PATIENCE = 10; LR_PATIENCE = 3; LR_FACTOR = 0.3
PAIR_BATCH = 64; TRAIN_PAIRS_PER_EPOCH = 12000; VAL_PAIRS = None; EMBED_BATCH = 512; TIMING_REPEATS = 30; TRAIN_AUGMENTATION = True
QUICK_SMOKE_TEST = False
if QUICK_SMOKE_TEST:
    EPOCHS = 2; TRAIN_PAIRS_PER_EPOCH = 384; VAL_PAIRS = 384; REPS_TO_RUN = REPS[:1]; KDIR = KDIR / '_smoke_test'; FIG = KDIR / 'figures'
FIG.mkdir(parents=True, exist_ok=True); (KDIR / 'runs').mkdir(parents=True, exist_ok=True); LOG = L.RunLog(KDIR / 'FPP13_run_log.csv')
tf.keras.mixed_precision.set_global_policy('float32')
MOD_PATH = Path('/content/fpp13_minutiae.py'); MOD_PATH.write_text(r'''# fpp13_minutiae.py — the FPP09 minutiae module, unchanged (FPP13 trains the encoder together with the Siamese matcher)
import numpy as np
SETUP = dict(quality=False, k=16, depth=2, N=128, aggregate='max', crop=False)          # architecture ablation: pilot defaults (FPM07), no quality input
ST = dict(d=256, heads=4, blocks=3, ff=512)                 # Set Transformer: model width, heads, self-attention blocks, feed-forward width
EDGE1, EDGE2, POINT = [64, 128], [256, 256], 512; EMBEDDING_DIM = 1024; DROPOUT = 0.3; GEO_SCALE = 4.0
AUG = dict(rotation_deg=5.0, shift_px=10.0, dropout=0.10, position_sigma_px=2.0, angle_sigma_deg=5.0, p_original=0.25, crop_area_min=0.70)
# placement (as the FPP01 image recipe: rotation +-5 deg, shift +-3 % of the image ~ +-10 px at 500 ppi) + FingerJetFX noise (FPM08: 10 % minutia dropout, 2 px / 5 deg jitter); 25 % of draws stay original (as FPP01 mix_versions)
COORD_HALF = 112.0      # native 500 ppi pixels, centred on the image, / 112 (one physical scale for every sensor)

def load_minutiae(h5file):
    import h5py
    out = []
    with h5py.File(h5file, 'r') as h:
        g = h['minutiae']; X, Y, TH, TY, Q = g['x'][:], g['y'][:], g['theta'][:], g['type'][:], g['quality'][:]; W, H = g['image_width'][:], g['image_height'][:]
        PPI = g['ppi'][:] if 'ppi' in g else np.full(len(X), 500)
    for i in range(len(X)):
        f = 500.0 / float(PPI[i] or 500)
        out.append({'x': ((np.asarray(X[i], np.float32) - W[i] / 2) * f).astype(np.float32), 'y': ((np.asarray(Y[i], np.float32) - H[i] / 2) * f).astype(np.float32),
                    'theta': np.deg2rad(np.asarray(TH[i], np.float32)).astype(np.float32), 'type': np.clip(np.asarray(TY[i], np.int64), 0, 2),
                    'quality': np.asarray(Q[i], np.float32)})
    return out

def augment(m, rng, crop=False):
    # training only: with p 0.25 the original; else whole-print rotation + shift (placement), then minutia dropout and position / angle jitter (extraction noise)
    if len(m['x']) == 0 or rng.random() < AUG['p_original']: return m
    m = rotate_translate(m, np.deg2rad(rng.uniform(-AUG['rotation_deg'], AUG['rotation_deg'])), rng.uniform(-AUG['shift_px'], AUG['shift_px']), rng.uniform(-AUG['shift_px'], AUG['shift_px']))
    x, y, t, ty, q = m['x'].copy(), m['y'].copy(), m['theta'].copy(), m['type'].copy(), m['quality'].copy()
    if crop and len(x) >= 8:
        # partial crop (FPM08): a random window covering 70-100 % of the area the minutiae occupy; minutiae outside it are removed (kept only if >= 5 remain)
        x0, x1, y0, y1 = x.min(), x.max(), y.min(), y.max(); s = np.sqrt(rng.uniform(AUG['crop_area_min'], 1.0)); w, h = (x1 - x0) * s, (y1 - y0) * s
        cx, cy = rng.uniform(x0, x1 - w) if x1 - x0 > w else x0, rng.uniform(y0, y1 - h) if y1 - y0 > h else y0
        inside = (x >= cx) & (x <= cx + w) & (y >= cy) & (y <= cy + h)
        if inside.sum() >= 5: x, y, t, ty, q = x[inside], y[inside], t[inside], ty[inside], q[inside]
    keep = rng.random(len(x)) >= AUG['dropout']
    if keep.sum() >= 5: x, y, t, ty, q = x[keep], y[keep], t[keep], ty[keep], q[keep]
    k = len(x); x = x + rng.normal(0, AUG['position_sigma_px'], k).astype(np.float32); y = y + rng.normal(0, AUG['position_sigma_px'], k).astype(np.float32)
    t = t + rng.normal(0, np.deg2rad(AUG['angle_sigma_deg']), k).astype(np.float32)
    return {'x': x.astype(np.float32), 'y': y.astype(np.float32), 'theta': t.astype(np.float32), 'type': ty, 'quality': q}

def to_set(m, n_max=128, quality=False):
    # (n_max, F + 1): x, y (/ COORD_HALF), cos, sin, type one-hot (3), [quality], mask; extra minutiae dropped by lowest quality
    F_ = 8 if quality else 7; n = len(m['x']); keep = np.argsort(-m['quality'], kind='stable')[:n_max] if n > n_max else np.arange(n); k = len(keep)
    s = np.zeros((n_max, F_ + 1), np.float32)
    s[:k, 0] = m['x'][keep] / COORD_HALF; s[:k, 1] = m['y'][keep] / COORD_HALF; s[:k, 2] = np.cos(m['theta'][keep]); s[:k, 3] = np.sin(m['theta'][keep])
    s[np.arange(k), 4 + m['type'][keep]] = 1.0
    if quality: s[:k, 7] = m['quality'][keep]
    s[:k, F_] = 1.0; return s

def rotate_translate(m, angle, tx, ty):
    c, s = np.cos(angle), np.sin(angle); x, y = m['x'], m['y']
    return {**m, 'x': (c * x - s * y + tx).astype(np.float32), 'y': (s * x + c * y + ty).astype(np.float32), 'theta': (m['theta'] + angle).astype(np.float32)}

def build_ls_encoder(cfg=SETUP, emb_dim=EMBEDDING_DIM):
    # local structure (FPM07 / FPM08): each minutia described by its k nearest neighbours in its own frame -> invariant to rotation and translation by design
    import tensorflow as tf
    Lyr = tf.keras.layers
    class MaskedPool(Lyr.Layer):
        def call(self, inputs):
            h, m = inputs; m3 = tf.cast(m, h.dtype)[..., None]; cnt = tf.reduce_sum(m3, 1)
            mean = tf.reduce_sum(h * m3, 1) / tf.maximum(cnt, 1.0); mx = tf.reduce_max(h + (m3 - 1.0) * 1e9, 1)
            return tf.concat([tf.where(cnt > 0, mx, tf.zeros_like(mx)), mean], -1)
    class KNN(Lyr.Layer):
        def __init__(self, k, **kw): super().__init__(**kw); self.k = int(k)
        def call(self, inputs):
            x, m = inputs; px, py = x[..., 0], x[..., 1]; n = tf.shape(x)[1]
            d2 = (px[:, None, :] - px[:, :, None]) ** 2 + (py[:, None, :] - py[:, :, None]) ** 2
            d2 = d2 + (1.0 - m[:, None, :]) * 1e9 + tf.eye(n, dtype=d2.dtype)[None] * 2e9
            return tf.math.top_k(-d2, k=self.k).indices
    class EdgeGeometry(Lyr.Layer):
        def __init__(self, quality=False, **kw): super().__init__(**kw); self.quality = bool(quality)
        def call(self, inputs):
            x, m, idx = inputs; xj = tf.gather(x, idx, batch_dims=1); xi = x[:, :, None, :]
            dx = xj[..., 0] - xi[..., 0]; dy = xj[..., 1] - xi[..., 1]; ci, si, cj, sj = xi[..., 2], xi[..., 3], xj[..., 2], xj[..., 3]
            u = dx * ci + dy * si; v = -dx * si + dy * ci; dist = tf.sqrt(u * u + v * v + 1e-12)
            ti = tf.broadcast_to(xi[..., 4:7], tf.shape(xj[..., 4:7])); valid = tf.gather(m, idx, batch_dims=1) * m[:, :, None]
            parts = [tf.stack([GEO_SCALE * u, GEO_SCALE * v, GEO_SCALE * dist, cj * ci + sj * si, sj * ci - cj * si], -1), ti, xj[..., 4:7]]
            if self.quality: parts += [tf.broadcast_to(xi[..., 7:8], tf.shape(xj[..., 7:8])), xj[..., 7:8]]
            return tf.concat(parts + [valid[..., None]], -1)
    class MaskedAttnK(Lyr.Layer):
        # neighbour attention: softmax over the k neighbours of a learned score per edge (padding masked); weighted sum of the edge features
        def call(self, inputs):
            e, s, valid = inputs; logits = s[..., 0] + (valid - 1.0) * 1e9; w = tf.nn.softmax(logits, axis=2)
            out = tf.reduce_sum(w[..., None] * e, 2); return tf.where(tf.reduce_sum(valid, 2, keepdims=True) > 0, out, tf.zeros_like(out))
    def aggregate(e, valid, name):
        if cfg.get('aggregate', 'max') == 'max': return MaskedMaxK(name=name)([e, valid])
        return MaskedAttnK(name=name)([e, Lyr.Dense(1, name=f'{name}_attn_score')(e), valid])
    class GatherNeighbours(Lyr.Layer):
        def call(self, inputs): h, idx = inputs; return tf.gather(h, idx, batch_dims=1)
    class MaskedMaxK(Lyr.Layer):
        def call(self, inputs):
            e, valid = inputs; v = valid[..., None]; mx = tf.reduce_max(e + (v - 1.0) * 1e9, 2)
            return tf.where(tf.reduce_sum(v, 2) > 0, mx, tf.zeros_like(mx))
    F_ = 8 if cfg['quality'] else 7; k = cfg['k']; inp = tf.keras.Input((cfg['N'], F_ + 1), dtype=tf.float32, name='minutiae_set')
    x = Lyr.Lambda(lambda z: z[..., :F_], name='features')(inp); m = Lyr.Lambda(lambda z: z[..., F_], name='mask')(inp)
    idx = KNN(k, name='knn')([x, m]); eg = EdgeGeometry(cfg['quality'], name='edge_geometry')([x, m, idx])
    geo = Lyr.Lambda(lambda z: z[..., :-1], name='edge_features')(eg); valid = Lyr.Lambda(lambda z: z[..., -1], name='edge_valid')(eg)
    e = geo
    for i, u in enumerate(EDGE1): e = Lyr.Dense(u, activation='relu', name=f'edge1_dense_{i}')(e)   # explicit names: weights load by structure in any process
    h = aggregate(e, valid, 'local_structure_1'); hs = [h]
    for layer in range(2, cfg['depth'] + 1):
        hj = GatherNeighbours(name=f'gather_{layer}')([h, idx]); hi = Lyr.Lambda(lambda z: tf.repeat(z[:, :, None, :], k, axis=2), name=f'tile_{layer}')(h)
        e2 = Lyr.Concatenate(name=f'edge_concat_{layer}')([hi, Lyr.Subtract(name=f'edge_diff_{layer}')([hj, hi]), geo])
        for i, u in enumerate(EDGE2): e2 = Lyr.Dense(u, activation='relu', name=f'edge{layer}_dense_{i}')(e2)
        h = aggregate(e2, valid, f'local_structure_{layer}'); hs.append(h)
    h = Lyr.Dense(POINT, activation='relu', name='point_dense')(Lyr.Concatenate(name='local_concat')(hs) if len(hs) > 1 else hs[0]); pooled = MaskedPool(name='pool')([h, m])
    f = Lyr.Dropout(DROPOUT, name='head_dropout')(pooled); e = Lyr.Dense(emb_dim, use_bias=False, name='embedding_dense')(f); e = Lyr.BatchNormalization(name='embedding_bn')(e)
    e = Lyr.Lambda(lambda z: tf.math.l2_normalize(z, axis=1), name='embedding')(e)
    return tf.keras.Model(inp, e, name='minutiae_local_structure_encoder')

def build_st_encoder(cfg=SETUP, st=ST, emb_dim=EMBEDDING_DIM):
    # Set Transformer (Lee et al. 2019): per-minutia features -> pre-norm self-attention blocks over all minutiae (padding masked) -> attention pooling (PMA, 1 seed) + masked mean
    import tensorflow as tf
    Lyr = tf.keras.layers
    class Seed(Lyr.Layer):
        def build(self, shape): self.s = self.add_weight(name='seed', shape=(1, 1, st['d']), initializer='glorot_uniform')
        def call(self, h): return tf.tile(self.s, [tf.shape(h)[0], 1, 1])
    class MaskedMean(Lyr.Layer):
        def call(self, inputs):
            h, m = inputs; m3 = m[..., None]; return tf.reduce_sum(h * m3, 1) / tf.maximum(tf.reduce_sum(m3, 1), 1.0)
    F_ = 8 if cfg['quality'] else 7; inp = tf.keras.Input((cfg['N'], F_ + 1), dtype=tf.float32, name='minutiae_set')
    x = Lyr.Lambda(lambda z: z[..., :F_], name='features')(inp); m = Lyr.Lambda(lambda z: z[..., F_], name='mask')(inp)
    key_mask = Lyr.Lambda(lambda z: tf.tile(z[:, None, :] > 0.5, [1, tf.shape(z)[1], 1]), name='key_mask')(m)       # (B, N, N): attend to real minutiae only
    seed_mask = Lyr.Lambda(lambda z: z[:, None, :] > 0.5, name='seed_mask')(m)                                       # (B, 1, N)
    h = Lyr.Dense(st['d'], name='input_proj')(x)
    for b in range(st['blocks']):
        a = Lyr.LayerNormalization(name=f'sab{b}_ln1')(h)
        a = Lyr.MultiHeadAttention(st['heads'], st['d'] // st['heads'], name=f'sab{b}_mha')(a, a, attention_mask=key_mask)
        h = Lyr.Add(name=f'sab{b}_res1')([h, a])
        f = Lyr.LayerNormalization(name=f'sab{b}_ln2')(h); f = Lyr.Dense(st['ff'], activation='gelu', name=f'sab{b}_ff1')(f); f = Lyr.Dense(st['d'], name=f'sab{b}_ff2')(f)
        h = Lyr.Add(name=f'sab{b}_res2')([h, f])
    h = Lyr.LayerNormalization(name='final_ln')(h)
    q = Seed(name='pma_seed')(h); pma = Lyr.MultiHeadAttention(st['heads'], st['d'] // st['heads'], name='pma_mha')(q, h, attention_mask=seed_mask)
    pooled = Lyr.Concatenate(name='pool')([Lyr.Flatten(name='pma_flat')(pma), MaskedMean(name='masked_mean')([h, m])])
    f = Lyr.Dropout(DROPOUT, name='head_dropout')(pooled); e = Lyr.Dense(emb_dim, use_bias=False, name='embedding_dense')(f); e = Lyr.BatchNormalization(name='embedding_bn')(e)
    e = Lyr.Lambda(lambda z: tf.math.l2_normalize(z, axis=1), name='embedding')(e)
    return tf.keras.Model(inp, e, name='minutiae_set_transformer_encoder')

BUILDERS = {'set_transformer': build_st_encoder, 'local_structure': build_ls_encoder}
def build_encoder(arch): return BUILDERS[arch]()

def arm_cfg(arm):
    # 'k16' (FPP06) / 'k8' / 'k24' / 'k32' / 'k<K>_attention'
    k = int(arm.split('_')[0][1:]); return dict(SETUP, k=k, aggregate='attention' if arm.endswith('_attention') else 'max')
def build_arm(arm): return build_ls_encoder(arm_cfg(arm))

def transfer(src, dst):
    # copy every weight of `src` into the layer of the same name in `dst`; a layer that only GREW its input (quality inputs, a third layer feeding the
    # concatenation) gets the old weights in its first rows and ZEROS in the new rows -> dst starts with exactly the same function as src
    names = {l.name: l for l in dst.layers}; copied, grown = [], []
    for l in src.layers:
        if not l.weights: continue
        d = names.get(l.name)
        if d is None: raise RuntimeError(f'layer {l.name} missing in the new model')
        ws_, wd = l.get_weights(), d.get_weights(); new = []
        for a, b in zip(ws_, wd):
            if a.shape == b.shape: new.append(a)
            elif a.ndim == 2 and b.ndim == 2 and a.shape[1] == b.shape[1] and b.shape[0] > a.shape[0]:
                z = np.zeros_like(b); z[:a.shape[0]] = a; new.append(z)
            else: raise RuntimeError(f'cannot transfer {l.name}: {a.shape} -> {b.shape}')
        d.set_weights(new); (grown if any(a.shape != b.shape for a, b in zip(ws_, wd)) else copied).append(l.name)
    fresh = [l.name for l in dst.layers if l.weights and l.name not in {s.name for s in src.layers}]
    return {'layers_copied': len(copied), 'layers_grown_zero_rows': grown, 'layers_new': fresh}
'''); sys.path.insert(0, '/content')
import fpp13_minutiae as M; importlib.reload(M)

def winner_dirs(kernel):
    mf = json.loads((P.kernel_dir(ws, kernel) / 'winner_manifest.json').read_text())
    return {r: (Path(ws['output']) / f['pair_scores.npz']).parent for r, f in mf['files'].items()}, mf
IMG_RUN, mf5 = winner_dirs('FPP05'); MIN_RUN, mf9 = winner_dirs('FPP09')
M_CFG = dict(M.SETUP, **mf9['config'])
ENC_RUN = {r: FPP11_DIR / 'runs' / r / 'best_plus_mm' for r in REPS}; SIA_RUN = {r: FPP12_DIR / 'runs' / r / 'siamese' for r in REPS}
for name_, dd in (('FPP11 best_plus_mm', ENC_RUN), ('FPP12 siamese', SIA_RUN)):
    miss = [r for r in REPS_TO_RUN if not L.run_complete(dd[r])]
    if miss: raise RuntimeError(f'{name_} is not finished for {miss}: run it first')
IMG_LABEL = 'Image vs image (FPP04L winner, after re-enrolment)'; MIN_LABEL = 'Minutiae vs minutiae (FPP09 winner, old template)'; BOZ = 'FingerJetFX + BOZORTH3 (FPP03)'
CONFIG = dict(kernel=KERNEL, kernel_name=P.FPP_KERNELS[KERNEL], protocol=P.PROTOCOL_TABLE, arms=ARMS, arm_config=ARM_CFG, candidates_also=PREV,
              deployment='batch migration: minutiae encoder + Siamese tower -> migration template; first login: image model + Siamese tower -> cosine; on a match re-enrol with the normal image embedding',
              start='FPP11 best_plus_mm encoder + FPP12 siamese tower (same repetition), checked against their saved outputs', minutiae_config=M_CFG,
              loss=f'cross-modal BCE both directions + {LAMBDA_PULL} x (1 - cos(minutiae output, image output of the same capture)); joint_mm: + {LAMBDA_MM} x minutiae-vs-minutiae pair BCE',
              augmentation='FPP09 minutiae augmentation incl. partial crop (clean image embeddings)', epochs=EPOCHS, weight_decay=WEIGHT_DECAY, clipnorm=CLIPNORM,
              patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR, pair_batch=PAIR_BATCH, train_pairs_per_epoch=TRAIN_PAIRS_PER_EPOCH, sampler='source-balanced DB1 / DB2 / DB1xDB2',
              early_stopping='first-login cross-modal ' + P.SELECTION_RULE, threshold=P.THRESHOLD_RULE + ' (cross-modal scores)', decision=P.DECISION_RULE,
              local_runs=str(P.LOCAL_RUNS), repetitions=REPS, quick_smoke_test=QUICK_SMOKE_TEST, fpp_lib_version=P.VERSION)
print(json.dumps(CONFIG, indent=2, default=str))

## 2. Saved image embeddings, minutiae templates, pair sequence (fresh augmentation every epoch)

In [ ]:
I_EMB = {r: {d: L.l2n(v.astype(np.float32)) for d, v in L.load_npz(IMG_RUN[r] / 'embeddings_float32.npz').items()} for r in REPS}
MINU = {d: M.load_minutiae(L.local_copy(ws['templates'] / f'{d}_fingerjetfx_templates.h5')) for d in L.DATASETS}
MOLF = ('MOLF_DB1', 'MOLF_DB2'); MINU_MOLF = MINU['MOLF_DB1'] + MINU['MOLF_DB2']; OFFSET = {'MOLF_DB1': 0, 'MOLF_DB2': len(MINU['MOLF_DB1'])}
SETS = {d: np.stack([M.to_set(m, quality=M_CFG['quality']) for m in MINU[d]]) for d in L.DATASETS}; SETS['MOLF'] = np.concatenate([SETS[d] for d in MOLF])
for r in REPS:
    for d in L.DATASETS:
        if len(I_EMB[r][d]) != len(MINU[d]): raise RuntimeError(f'{r} {d}: image embeddings and templates differ in count')
I_MOLF = {r: np.concatenate([I_EMB[r][d] for d in MOLF]) for r in REPS}
PROTO = {r: P.load_pairs(ws, r) for r in REPS}; print('FPP01 pair files verified for', len(REPS), 'repetitions')
IMG_MOLF = SETS['MOLF']             # minutiae-side inputs of the validation templates (the callback embeds IMG_MOLF[bv.indices])

class CrossPairSeq(tf.keras.utils.Sequence):
    # source-balanced pairs: freshly augmented minutiae sets of a and b + frozen image embeddings of a and b
    def __init__(self, sampler, seed, I_molf, workers=4):
        super().__init__(workers=workers, use_multiprocessing=False, max_queue_size=8); self.s = sampler; self.seed = seed; self.I = I_molf; self.epoch = 0
    def __len__(self): return len(self.s)
    def _x(self, idx, rng):
        if TRAIN_AUGMENTATION: return np.stack([M.to_set(M.augment(MINU_MOLF[int(i)], rng, M_CFG['crop']), quality=M_CFG['quality']) for i in idx])
        return SETS['MOLF'][np.asarray(idx)]
    def __getitem__(self, b):
        ia, ib, y = self.s.batch(b); ia, ib = np.asarray(ia), np.asarray(ib); rng = np.random.default_rng([self.seed, self.epoch, b])
        return (self._x(ia, rng), self._x(ib, rng), self.I[ia], self.I[ib]), y
    def on_epoch_end(self): self.s.on_epoch_end(); self.epoch += 1

## 3. Models: encoder → Siamese tower (minutiae side), Siamese tower (image side); training model; validation
`F` = encoder followed by the tower with the minutiae offset (the migration model); `G` = the tower with the image offset (first-login probe side).

In [ ]:
class EpochClock(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None): self.t0 = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None: logs['epoch_seconds'] = time.perf_counter() - self.t0
class BalancedEERCallback(tf.keras.callbacks.Callback):
    # after each epoch: embed the validation images, score all validation pairs, source-balanced EER -> best checkpoint / LR drop / early stop (fpp_lib rule)
    def __init__(self, encoder, bv, ckpt, hist_csv):
        super().__init__(); self.encoder = encoder; self.bv = bv; self.ckpt = ckpt; self.hist_csv = hist_csv; self.x = IMG_MOLF[bv.indices]; self.rows = []
        self.stop = P.BalancedEERStopping(PATIENCE, LR_PATIENCE, LR_FACTOR)
    def _lr(self): return float(np.asarray(self.model.optimizer.learning_rate))
    def _set_lr(self, v):
        try: self.model.optimizer.learning_rate.assign(v)
        except AttributeError: self.model.optimizer.learning_rate = v
    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}; t0 = time.perf_counter()
        m = self.bv.score(L.l2n(T.embed(self.encoder, self.x, EMBED_BATCH))); logs.update(m); lr = self._lr(); d = self.stop.update(epoch, m['val_balanced_eer'], m['val_balanced_auc'])
        if d['improved']: self.model.save_weights(str(self.ckpt))
        if d['reduce_lr'] and lr > self.stop.min_lr: self._set_lr(max(lr * LR_FACTOR, self.stop.min_lr))
        if d['stop']: self.model.stop_training = True
        logs['validation_seconds'] = time.perf_counter() - t0
        self.rows.append({'epoch': epoch + 1, **{k: float(np.asarray(v)) for k, v in logs.items() if np.asarray(v).size == 1}, 'lr': lr, 'improved': d['improved']}); pd.DataFrame(self.rows).to_csv(self.hist_csv, index=False)
        print(f"  epoch {epoch + 1}: val balanced EER {100 * m['val_balanced_eer']:.2f} % | AUC {100 * m['val_balanced_auc']:.2f} % | "
              + ' '.join(f"{n.split('_')[0]} {100 * m[f'val_eer_{n}']:.2f}" for n in P.VAL_SOURCES) + (' *best' if d['improved'] else '') + (' | LR down' if d['reduce_lr'] else '') + (' | stop' if d['stop'] else ''))


class CrossValidation(P.BalancedValidation):
    # a = enrolled minutiae (F output, embedded by the callback), b = probe image (G output, recomputed every epoch)
    def __init__(self, Pp, offset, I_molf, g, max_pairs=None):
        super().__init__(Pp, offset, max_pairs=max_pairs); self.I_raw = np.asarray(I_molf, np.float32)[self.indices]; self.g = g; self.I = None
    def score(self, emb):
        self.I = L.l2n(T.embed(self.g, self.I_raw, EMBED_BATCH)); return super().score(emb)
    def pair_score(self, E, ia, ib): return np.sum(E[ia] * self.I[ib], axis=1)

Lyr = tf.keras.layers
class ModalityOffset(Lyr.Layer):
    def build(self, shape): self.b = self.add_weight(name='offset', shape=(int(shape[-1]),), initializer='zeros')
    def call(self, x): return x + self.b

def build_siamese():
    # the FPP12 'siamese' structure and layer names, so its saved tower weights load as they were saved
    xm = tf.keras.Input((M.EMBEDDING_DIM,), name='minutiae_embedding'); xi = tf.keras.Input((M.EMBEDDING_DIM,), name='image_embedding')
    l2 = lambda n: Lyr.Lambda(lambda z: tf.math.l2_normalize(z, axis=1), name=n)
    om = ModalityOffset(name='minutiae_offset'); oi = ModalityOffset(name='image_offset')
    hd = Lyr.Dense(ADAPTER_HIDDEN, activation='gelu', name='shared_hidden'); dr = Lyr.Dropout(ADAPTER_DROPOUT, name='shared_dropout'); out = Lyr.Dense(M.EMBEDDING_DIM, name='shared_out')
    f = tf.keras.Model(xm, l2('minutiae_shared_embedding')(out(dr(hd(om(xm))))), name='siamese_minutiae'); g = tf.keras.Model(xi, l2('image_shared_embedding')(out(dr(hd(oi(xi))))), name='siamese_image')
    return f, g

def build_models(rep, train_encoder):
    # encoder from FPP11 best_plus_mm, tower from FPP12 siamese; both checked against their saved outputs
    enc = M.build_ls_encoder(M_CFG); enc.load_weights(str(ENC_RUN[rep] / 'encoder.weights.h5'))
    Es = L.load_npz(ENC_RUN[rep] / 'embeddings_float32.npz')['MOLF_DB1'][:32]; E1 = L.l2n(T.embed(enc, SETS['MOLF_DB1'][:32], 32)); c1 = float(np.min(np.sum(Es * E1, 1)))
    ft, gt = build_siamese(); ft.load_weights(str(SIA_RUN[rep] / 'minutiae_tower.weights.h5')); gt.load_weights(str(SIA_RUN[rep] / 'image_tower.weights.h5'))
    St = L.load_npz(SIA_RUN[rep] / 'embeddings_float32.npz')['MOLF_DB1'][:32]; S1 = L.l2n(T.embed(ft, Es, 32)); c2 = float(np.min(np.sum(St * S1, 1)))
    if min(c1, c2) < 0.999: raise RuntimeError(f'{rep}: reloaded start weights do not reproduce the saved outputs (encoder {c1:.5f}, tower {c2:.5f})')
    enc.trainable = bool(train_encoder)
    x = tf.keras.Input(SETS['MOLF'].shape[1:], name='minutiae_set'); F = tf.keras.Model(x, ft(enc(x, training=None if train_encoder else False)), name='migration_model')
    return enc, ft, gt, F, {'start_encoder': L.rel(ws, ENC_RUN[rep]), 'start_tower': L.rel(ws, SIA_RUN[rep]), 'start_check_encoder': round(c1, 6), 'start_check_tower': round(c2, 6)}

class JointModel(tf.keras.Model):
    # F = encoder + Siamese tower (minutiae side), G = Siamese tower (image side); loss = cross-modal BCE + pull (+ minutiae-vs-minutiae BCE)
    def __init__(self, F, G, mm=False, **kw):
        super().__init__(**kw); self.F = F; self.G = G; self.mm = bool(mm); self.bce = tf.keras.losses.BinaryCrossentropy()
        self.calib = T.CosineCalibration(name='same_finger_probability'); self.calib_mm = T.CosineCalibration(name='minutiae_pair_probability')
        self.t_loss = tf.keras.metrics.Mean(name='loss'); self.t_bce = tf.keras.metrics.Mean(name='bce'); self.t_pull = tf.keras.metrics.Mean(name='pull_loss'); self.t_mm = tf.keras.metrics.Mean(name='mm_loss')
        self.t_auc = tf.keras.metrics.AUC(name='verification_auc')
    @property
    def metrics(self): return [self.t_loss, self.t_bce, self.t_pull, self.t_mm, self.t_auc]
    def call(self, inputs, training=False):
        return self.calib(tf.reduce_sum(self.F(inputs[0], training=training) * self.G(inputs[1], training=training), axis=1))
    def train_step(self, data):
        (xa, xb, Ia, Ib), y = data; y = tf.cast(tf.reshape(y, [-1]), tf.float32)
        with tf.GradientTape() as tape:
            ma = self.F(xa, training=True); mb = self.F(xb, training=True); ia = self.G(tf.cast(Ia, tf.float32), training=True); ib = self.G(tf.cast(Ib, tf.float32), training=True)
            pa = self.calib(tf.reduce_sum(ma * ib, axis=1)); pb = self.calib(tf.reduce_sum(mb * ia, axis=1)); l_bce = 0.5 * (self.bce(y, pa) + self.bce(y, pb))
            l_pull = 0.5 * (tf.reduce_mean(1.0 - tf.reduce_sum(ma * ia, axis=1)) + tf.reduce_mean(1.0 - tf.reduce_sum(mb * ib, axis=1)))
            l_mm = self.bce(y, self.calib_mm(tf.reduce_sum(ma * mb, axis=1))) if self.mm else tf.constant(0.0)
            loss = l_bce + LAMBDA_PULL * l_pull + LAMBDA_MM * l_mm
        tv = list({id(v_): v_ for v_ in self.trainable_variables}.values())          # the tower is shared by F and G: each variable once
        self.optimizer.apply_gradients([(g_, v_) for g_, v_ in zip(tape.gradient(loss, tv), tv) if g_ is not None])
        for m_, v_ in ((self.t_loss, loss), (self.t_bce, l_bce), (self.t_pull, l_pull), (self.t_mm, l_mm)): m_.update_state(v_)
        self.t_auc.update_state(tf.concat([y, y], 0), tf.concat([pa, pb], 0)); return {m_.name: m_.result() for m_ in self.metrics}

def build_joint(F, G, lr, mm):
    m = JointModel(F, G, mm, name='joint_migration'); m((SETS['MOLF'][:1], np.zeros((1, M.EMBEDDING_DIM), np.float32))); m.calib_mm(tf.zeros((1,), tf.float32)); m.built = True
    m.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=WEIGHT_DECAY, clipnorm=CLIPNORM)); return m
print('models ready | arms:', ARM_CFG)

## 4. One run = one (arm, repetition)
Saved per run: encoder and tower weights, migration templates (`F` output), first-login probe side (`G` output), cross-modal scores (main), and reported only: minutiae vs minutiae and image enrolled vs minutiae probe through the tower.

In [ ]:
def scores(Ea, Eb, Pp): return {n: (np.asarray(Pp[n]['label'], int), L.pair_scores(Ea[da], Eb[db], Pp[n]).astype(np.float64)) for n, (da, db, _) in P.PAIR_SETS.items()}

def run_arm(arm, rep):
    rd = KDIR / 'runs' / rep / arm
    if L.run_complete(rd): print(arm, rep, 'complete -> skipped'); return
    cfg = ARM_CFG[arm]; lrd = P.local_run_dir(KERNEL + ('_smoke' if QUICK_SMOKE_TEST else ''), rep, arm); proto, Pp = PROTO[rep]; seed = proto['seed']; t = L.timer()
    tf.keras.backend.clear_session(); gc.collect(); T.set_all_seeds(seed)
    enc, ft, gt, F, start = build_models(rep, cfg['train_encoder']); model = build_joint(F, gt, cfg['lr'], cfg['mm'])
    params = {'parameters_encoder': T.count_params(enc)['parameters_total'], 'parameters_tower': T.count_params(gt)['parameters_total'],
              'parameters_trainable': int(sum(np.prod(v.shape) for v in {id(v): v for v in model.trainable_variables}.values()))}
    ssf = float(proto['manifest']['same_subject_impostor_fraction'])
    seq = CrossPairSeq(P.MixedPairs(P.sources(Pp, P.TRAIN_SOURCES, OFFSET), PAIR_BATCH, True, seed, TRAIN_PAIRS_PER_EPOCH, ssf), seed, I_MOLF[rep])
    bv = CrossValidation(Pp, OFFSET, I_MOLF[rep], gt, max_pairs=VAL_PAIRS)
    LOG.add(step='train_start', arm=arm, repetition=rep, **cfg, **start, train_composition=json.dumps(seq.s.composition), validation_pairs=json.dumps(bv.composition), **params)
    ckpt = lrd / 'best_joint.weights.h5'; hist_csv = lrd / 'training_history.csv'; fit_marker = lrd / 'fit_complete.json'
    if ckpt.is_file() and fit_marker.is_file(): train_s = json.loads(fit_marker.read_text())['training_seconds']
    else:
        model.fit(seq, epochs=EPOCHS, callbacks=[EpochClock(), BalancedEERCallback(F, bv, ckpt, hist_csv)], verbose=2); train_s = t(); fit_marker.write_text(json.dumps({'training_seconds': train_s}))
    model.load_weights(ckpt); hist = pd.read_csv(hist_csv)
    b_i, _ = P.BalancedEERStopping.replay(hist, EPOCHS, patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR); best = hist.iloc[b_i]
    emb_m = {d: L.l2n(T.embed(F, SETS[d], EMBED_BATCH)) for d in L.DATASETS}; emb_i = {d: L.l2n(T.embed(gt, I_EMB[rep][d], EMBED_BATCH)) for d in L.DATASETS}
    enc.save_weights(lrd / 'encoder.weights.h5'); ft.save_weights(lrd / 'minutiae_tower.weights.h5'); gt.save_weights(lrd / 'image_tower.weights.h5')
    np.savez_compressed(lrd / 'embeddings_float32.npz', **{d: e.astype(np.float32) for d, e in emb_m.items()})
    np.savez_compressed(lrd / 'image_side_embeddings_float32.npz', **{d: e.astype(np.float32) for d, e in emb_i.items()})
    m_batch, m_one = T.time_encoder(F, SETS['MOLF_DB1'][:64], TIMING_REPEATS); i_batch, i_one = T.time_encoder(gt, I_EMB[rep]['MOLF_DB1'][:64], TIMING_REPEATS)
    extra = {'trained': True, **cfg, **start, 'best_epoch': int(best.epoch), 'epochs_run': int(len(hist)), 'val_balanced_eer_at_best': float(best.val_balanced_eer), 'training_seconds': train_s,
             'seconds_per_epoch': float(hist['epoch_seconds'].mean()) if 'epoch_seconds' in hist else float('nan'),
             'migration_ms_per_template_gpu_batch64': m_batch, 'image_tower_ms_single_login_gpu': i_one, **params}
    (lrd / 'model_config.json').write_text(json.dumps({'arm': arm, **cfg, 'loss': CONFIG['loss'], 'minutiae_config': M_CFG, 'module': 'fpp13_minutiae.py',
                                                        'weights': ['encoder.weights.h5', 'minutiae_tower.weights.h5', 'image_tower.weights.h5'], 'start': start}, indent=2, default=str))
    same = float(np.mean(np.sum(np.concatenate([emb_m[d] for d in MOLF]) * np.concatenate([emb_i[d] for d in MOLF]), 1)))
    S = scores(emb_m, emb_i, Pp); rows, thr = P.evaluate(S, KERNEL, arm, rep, {'comparison': 'minutiae enrolled vs image probe (first login)', 'cos_same_capture': same, **extra})
    for fname, S2, what in (('metrics_minutiae_vs_minutiae.csv', scores(emb_m, emb_m, Pp), 'minutiae enrolled vs minutiae probe (through the migration model)'),
                            ('metrics_image_enrol_minutiae_probe.csv', scores(emb_i, emb_m, Pp), 'image enrolled vs minutiae probe (through the tower)')):
        pd.DataFrame(P.evaluate(S2, KERNEL, arm, rep, {'comparison': what})[0]).to_csv(lrd / fname, index=False)
    P.save_run(lrd, S, rows, thr, {'kernel': KERNEL, 'arm': arm, 'repetition': rep, 'seed': seed, 'cos_same_capture': same, **extra})
    n_files = P.publish_run(lrd, rd, skip=('fit_complete.json', 'best_joint.weights.h5'))
    P.fig_training_balanced(hist, f'{arm} {rep[:6]} — stopping on first-login source-balanced validation EER', FIG / 'runs' / rep / f'{arm}_training.png')
    v = [r_ for r_ in rows if r_['set'] == P.SELECTION_SET][0]
    LOG.add(step='train_done', arm=arm, repetition=rep, status='ok', minutes=round(t() / 60, 1), files_to_drive=n_files, cos_same_capture=round(same, 4),
            val_balanced_eer=round(100 * v['eer'], 2), val_balanced_auc=round(100 * v['roc_auc'], 2), val_balanced_f1=round(100 * v['f1_macro'], 2))
    print(f"{arm} {rep[:6]}: first-login validation EER {100 * v['eer']:.2f} % | cos(same capture) {same:.3f}")

## 5. Run every arm for every repetition

In [ ]:
for arm in ARMS_TO_RUN:
    for rep in REPS_TO_RUN:
        try: run_arm(arm, rep)
        except (tf.errors.ResourceExhaustedError, tf.errors.InternalError) as exc:
            LOG.add(step='train_failed', arm=arm, repetition=rep, status='gpu_error', error=repr(exc)[:400]); print('GPU error', arm, rep, repr(exc)[:200]); tf.keras.backend.clear_session(); gc.collect()
        except Exception as exc:
            LOG.add(step='train_failed', arm=arm, repetition=rep, status='error', error=repr(exc)[:400]); print('FAILED', arm, rep, exc)
print(LOG.frame().tail(15).to_string())

## 6. Decision on VALIDATION only (first-login match); references; test and FingerDOS reported, not used; migration path

In [ ]:
complete = [a for a in ARMS if all(L.run_complete(KDIR / 'runs' / r / a) for r in REPS_TO_RUN)]
if not complete: raise RuntimeError('no finished arm: run section 5 first')
RUN_DIRS = {a: {r: KDIR / 'runs' / r / a for r in REPS_TO_RUN} for a in complete}; RUN_DIRS[PREV] = {r: SIA_RUN[r] for r in REPS_TO_RUN}
CANDIDATES = complete + [PREV]
results = pd.concat([pd.read_csv(KDIR / 'runs' / r / a / 'metrics.csv').assign(kernel=a) for a in complete for r in REPS_TO_RUN], ignore_index=True)
def ref(label, kdir, arm):
    rows = pd.DataFrame(P.reference_rows(label, kdir, arm=arm, reps=REPS_TO_RUN)); rows['kernel'] = label; return rows
ref_prev = ref(PREV, FPP12_DIR, 'siamese')
rd0 = next(iter(IMG_RUN.values())); ref_img = ref(IMG_LABEL, rd0.parents[2], rd0.name); rd0 = next(iter(MIN_RUN.values())); ref_min = ref(MIN_LABEL, rd0.parents[2], rd0.name)
ref_b = ref(BOZ, P.kernel_dir(ws, 'FPP03'), 'BOZORTH3')
HIST = [pd.read_csv(RUN_DIRS[a][r] / 'training_history.csv').assign(arm=a, repetition=r) for a in complete for r in REPS_TO_RUN if (RUN_DIRS[a][r] / 'training_history.csv').is_file()]
histories = pd.concat(HIST, ignore_index=True) if HIST else pd.DataFrame()
ALLR = pd.concat([results, ref_prev, ref_img, ref_min, ref_b], ignore_index=True); SHOW = CANDIDATES + [IMG_LABEL, MIN_LABEL, BOZ]
WINNER, decision = P.decide(ALLR, CANDIDATES); val_table = P.validation_table(ALLR, SHOW)
print('DECISION RULE (first-login cross-modal match):', P.DECISION_RULE); display(decision.round(3)); print('Validation, mean ± std (%):'); display(val_table); print('WINNER (validation only):', WINNER)
def _cos(rd): return json.loads((rd / 'run_complete.json').read_text()).get('cos_same_capture', np.nan)
cos_same = pd.DataFrame([{'kernel': a, 'mean': np.mean(v_), 'std': np.std(v_, ddof=1)} for a in CANDIDATES for v_ in [[_cos(RUN_DIRS[a][r]) for r in REPS_TO_RUN]]]).set_index('kernel')
print('cos(minutiae output, image output of the same capture):'); display(cos_same.round(4))
paper_all, snum_all = P.paper_table(ALLR, SHOW, sets=['MOLF validation (balanced)', 'MOLF test (balanced)', 'DB1_test', 'DB2_test', 'DB1xDB2_test', 'FingerDOS_test'])
test_table = P.wide_table(snum_all, SHOW, sets=['MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'], metrics_=('roc_auc', 'eer', 'fmr', 'fnmr', 'f1_macro'))
print('Test and FingerDOS incl. F1 (frozen validation-EER threshold; reported, not used for the decision):'); display(test_table)
paired = pd.concat([P.paired_differences(ALLR, a, [x for x in CANDIDATES if x != a] + [IMG_LABEL, MIN_LABEL]) for a in complete], ignore_index=True)
def extra_rows(fname, label_fmt):
    out = [pd.read_csv(RUN_DIRS[a][r] / fname).assign(kernel=label_fmt.format(a)) for a in CANDIDATES for r in REPS_TO_RUN if (RUN_DIRS[a][r] / fname).is_file()]
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()
MM = extra_rows('metrics_minutiae_vs_minutiae.csv', '{} — minutiae vs minutiae'); RV = extra_rows('metrics_image_enrol_minutiae_probe.csv', '{} — image enrolled vs minutiae probe')
extra_all, extra_num = P.paper_table(pd.concat([MM, RV, ref_min, ref_img], ignore_index=True), list(MM.kernel.unique()) + list(RV.kernel.unique()) + [MIN_LABEL, IMG_LABEL],
                                      sets=['MOLF validation (balanced)', 'MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'])
print('Reported only — other combinations:'); display(extra_all)
def cell(frame, kernel, set_):
    if not len(frame) or 'set' not in frame.columns: return '—'
    v = frame[(frame.kernel == kernel) & (frame.set == set_)].eer * 100; return f'{v.mean():.2f} ± {v.std():.2f}' if len(v) else '—'
mat = []
for set_ in ('MOLF test (balanced)', 'FingerDOS_test'):
    mat += [{'set': set_, 'stage': 'first login (migration matcher)', 'enrolled': 'migrated minutiae', 'probe': 'image', 'EER (%)': cell(ALLR, WINNER, set_)},
            {'set': set_, 'stage': 'after re-enrolment', 'enrolled': 'image (unchanged model)', 'probe': 'image', 'EER (%)': cell(ref_img, IMG_LABEL, set_)},
            {'set': set_, 'stage': 'old system', 'enrolled': 'minutiae', 'probe': 'minutiae', 'EER (%)': f"FPP09 {cell(ref_min, MIN_LABEL, set_)} | BOZORTH3 {cell(ref_b, BOZ, set_)}"}]
migration_matrix = pd.DataFrame(mat); print(f'MIGRATION PATH — EER (%), winner {WINNER} (reported, not used for the decision):'); display(migration_matrix)
tcols = ['training_seconds', 'best_epoch', 'epochs_run', 'seconds_per_epoch', 'migration_ms_per_template_gpu_batch64', 'image_tower_ms_single_login_gpu', 'parameters_trainable']
tsrc = results[results.set == P.SELECTION_SET]; timing = tsrc.groupby('kernel')[[c_ for c_ in tcols if c_ in tsrc.columns]].agg(['mean', 'std']).reindex(complete)
timing.columns = [f'{a}_{b}' for a, b in timing.columns]; print('Training and migration time (reported only):'); display(timing.round(4))

## 6b. Bias check of the threshold (first-login match): 5 calibration hold-out subjects vs validation (MOLF test, balanced)

In [ ]:
CB = []
for a in CANDIDATES:
    for r in REPS_TO_RUN:
        rd = RUN_DIRS[a][r]; proto, Pp = PROTO[r]; Em = L.load_npz(rd / 'embeddings_float32.npz'); Ii = L.load_npz(rd / 'image_side_embeddings_float32.npz')
        E = {d: (Em[d], Ii[d]) for d in L.DATASETS}
        c_ = P.calibration_check(P.load_scores(rd), E, P.calibration_pairs(proto), pair_score=lambda Ea, Eb, p: L.pair_scores(Ea[0], Eb[1], p)); CB.append({'arm': a, 'repetition': r, **c_})
calib = pd.DataFrame(CB); cs = calib.drop(columns='repetition').groupby('arm').agg(['mean', 'std'])
calib_table = pd.DataFrame({'arm': cs.index,
    'threshold validation': [f"{a_:.3f} ± {b_:.3f}" for a_, b_ in zip(cs[('threshold_validation', 'mean')], cs[('threshold_validation', 'std')])],
    'threshold calibration (5 subj.)': [f"{a_:.3f} ± {b_:.3f}" for a_, b_ in zip(cs[('threshold_calibration', 'mean')], cs[('threshold_calibration', 'std')])],
    **{f'test {m.upper()} {tag} (%)': (100 * cs[(f'test_{m}_{tag}', 'mean')]).round(2).to_numpy() for m in ('fmr', 'fnmr', 'f1') for tag in ('validation', 'calibration')}}).reset_index(drop=True)
display(calib_table)

## 7. Figures

In [ ]:
MR = {a: [RUN_DIRS[a][r] for r in REPS_TO_RUN] for a in CANDIDATES}; MR[IMG_LABEL] = [IMG_RUN[r] for r in REPS_TO_RUN]; MR[MIN_LABEL] = [MIN_RUN[r] for r in REPS_TO_RUN]
P.standard_figures(FIG, MR, sets=P.PAPER_SETS)
P.fig_metric_bars(snum_all, SHOW, FIG / 'metric_bars_validation.png', sets=['MOLF validation (balanced)'], title='First-login validation (decision set) with references')
P.fig_metric_bars(snum_all, SHOW, FIG / 'metric_bars_test_reported.png', sets=['MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'], title='Test and FingerDOS (reported, not used for the decision)')
v = ALLR[ALLR.set == P.SELECTION_SET].groupby('kernel').eer.agg(['mean', 'std']) * 100; xs = [PREV] + complete; xl = ['FPP12 siamese'] + complete
fig, ax = plt.subplots(figsize=(9, 4)); ax.bar(range(len(xs)), v.loc[xs, 'mean'], yerr=v.loc[xs, 'std'], capsize=4, color=[P.COLORS[0] if a == WINNER else '0.7' for a in xs])
for lab, col, ls in ((IMG_LABEL, 'C2', '--'), (MIN_LABEL, 'C1', ':'), (BOZ, '0.4', '-.')):
    if lab in v.index: ax.axhline(v.loc[lab, 'mean'], color=col, ls=ls, lw=1.2, label=f"{lab}: {v.loc[lab, 'mean']:.2f} %")
ax.set_xticks(range(len(xs))); ax.set_xticklabels(xl); ax.set(ylabel='validation EER, source-balanced (%)', title='First-login match: frozen tower → augmented → joint → joint + minutiae term'); ax.legend(fontsize=7); ax.grid(alpha=.3, axis='y')
P._save(fig, FIG / 'joint_training_arms.png'); print('figures in', FIG)

## 8. Manifest and workbook

In [ ]:
if not QUICK_SMOKE_TEST and set(ARMS) <= set(complete):
    wf = ('minutiae_tower.weights.h5', 'image_tower.weights.h5') if WINNER == PREV else ('encoder.weights.h5', 'minutiae_tower.weights.h5', 'image_tower.weights.h5')
    files = tuple(k_ for k_ in wf + ('model_config.json', 'embeddings_float32.npz', 'image_side_embeddings_float32.npz', 'thresholds.json', 'pair_scores.npz') if all((RUN_DIRS[WINNER][r] / k_).is_file() for r in REPS_TO_RUN))
    manifest = {'kernel': KERNEL, 'kernel_name': P.FPP_KERNELS[KERNEL], 'winning_arm': WINNER, 'deployment': CONFIG['deployment'], 'minutiae_config': M_CFG,
                'encoder': 'FPP11 best_plus_mm (unchanged)' if WINNER == PREV or not ARM_CFG.get(WINNER, {}).get('train_encoder') else 'jointly trained (encoder.weights.h5 of the run)',
                'image_model': {r: L.rel(ws, d) for r, d in IMG_RUN.items()}, 'decision_rule': P.DECISION_RULE, 'decision': decision.to_dict('records'),
                'migration_path': migration_matrix.to_dict('records'), 'threshold_rule': P.THRESHOLD_RULE,
                'files': {r: {k_: L.rel(ws, RUN_DIRS[WINNER][r] / k_) for k_ in files} for r in REPS_TO_RUN}, 'module': 'fpp13_minutiae.py', 'fpp_lib_version': P.VERSION}
    (KDIR / 'winner_manifest.json').write_text(json.dumps(manifest, indent=2, default=str)); (KDIR / 'fpp13_minutiae.py').write_text(MOD_PATH.read_text()); print('winner_manifest.json written | winner:', WINNER)
SHEETS = {'Decision (validation)': decision, 'Validation table': val_table, 'Migration path': migration_matrix, 'Test+FingerDOS (reported)': test_table,
          'Other combinations (reported)': extra_all, 'cos same capture': cos_same.reset_index(), 'Time (reported)': timing.reset_index(), 'All sets (mean ± std)': paper_all,
          'Paired differences': paired, 'Calibration Bias Check': calib_table, 'Calibration (per rep)': calib, 'Results (per rep)': P.to_percent(ALLR),
          'Training history': histories, 'Config': pd.DataFrame([{k: json.dumps(v_, default=str) for k, v_ in CONFIG.items()}]).T.reset_index(), 'Run log': LOG.frame()}
L.write_workbook(KDIR / 'FPP13_Joint_Training_Results.xlsx', SHEETS)
for name, df in (('decision', decision), ('validation_table', val_table), ('migration_path', migration_matrix), ('test_fingerdos_reported', test_table),
                 ('other_combinations_reported', extra_all), ('paired_differences', paired), ('calibration_bias_check', calib_table), ('results_per_repetition', P.to_percent(ALLR))):
    df.to_csv(KDIR / f'FPP13_{name}.csv', index=False)
print('FPP13 complete | winner on validation:', WINNER)